# Домашнее задание № 4
Бинарные деревья и BST

## 1. Восстановление дерева из массива
Корень — `arr[0]`, дети узла с индексом `i` — `2*i+1` и `2*i+2`. `None` означает пустое место. Узлы храним через `TreeNode`. **O(n)** по времени и памяти.

In [1]:
class TreeNode:
    # Узел дерева хранит значение и ссылки на два дочерних узла.
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left  # None означает, что левого ребёнка нет.
        self.right = right  # Аналогично для правого ребёнка.


def build_tree(arr, i=0):
    # Один вызов строит поддерево с корнем в позиции i; по умолчанию начинаем с arr[0].
    # Если позиции нет или она пуста, узел не создаём — это условие остановки рекурсии.
    if i >= len(arr) or arr[i] is None:
        return None

    root = TreeNode(arr[i])
    # В позиционном представлении дети i находятся в 2*i+1 и 2*i+2.
    # Например, дети позиции 0 — позиции 1 и 2, дети позиции 1 — позиции 3 и 4.
    # Позиции None тоже занимают место; под отсутствующим узлом ветка не строится.
    root.left = build_tree(arr, 2 * i + 1)
    root.right = build_tree(arr, 2 * i + 2)
    # Дочерние вызовы вернули готовые поддеревья; связываем их через root и возвращаем его.
    return root

tree = build_tree([8, 9, 11, 7, 16, 3, 1])
assert tree.val == 8
assert tree.left.right.val == 16
assert build_tree([]) is None
assert build_tree([1, None, 2]).right.val == 2
print(tree.val, tree.left.val, tree.right.val)

8 9 11


## 2. Симметричное дерево
Рекурсивно сравниваем левое поддерево с правым: левый ребёнок одного узла должен совпадать с правым ребёнком другого. **O(n)** по времени, **O(h)** по памяти.

In [2]:
def is_symmetric(root):
    # mirror проверяет, являются ли два поддерева зеркальными отражениями друг друга.
    def mirror(a, b):
        if a is None or b is None:
            # Если оба None — ветви закончились симметрично; если только один — нет.
            return a is b
        # У непустых узлов должны совпасть значения, внешние ветви и внутренние ветви.
        # Поэтому левую ветвь a сравниваем с ПРАВОЙ ветвью b, и наоборот.
        # and завершает проверку сразу после первого несовпадения.
        return (a.val == b.val
                and mirror(a.left, b.right)
                and mirror(a.right, b.left))

    # Пустое дерево симметрично. У непустого сравниваем поддеревья слева и справа от корня.
    return root is None or mirror(root.left, root.right)

assert is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3]))
assert not is_symmetric(build_tree([1, 2, 2, None, 3, None, 3]))
assert is_symmetric(None)
print(is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])))

True


## 3. Минимальная глубина
Используем BFS: идём по уровням, и первый найденный лист будет ближайшим к корню. **O(n)** по времени, **O(w)** по памяти, где `w` — ширина дерева.

In [3]:
from collections import deque


def min_depth(root):
    if root is None:
        return 0  # У пустого дерева глубина нулевая; сам корень имеет глубину 1.

    # BFS обходит дерево по уровням: сначала глубина 1, затем 2, затем 3 и т.д.
    # В очереди храним пары (узел, его глубина); deque быстро удаляет элементы слева.
    queue = deque([(root, 1)])
    while queue:
        node, depth = queue.popleft()  # Берём самый давно добавленный узел.
        # Лист — узел БЕЗ ОБОИХ детей. Отсутствие только одного ребёнка не завершает путь.
        if node.left is None and node.right is None:
            # Это первый встреченный лист: более глубокие уровни ещё не обработаны.
            return depth
        # Детей ставим в конец очереди, за уже ожидающими узлами текущего уровня.
        # Каждый переход к ребёнку увеличивает длину пути от корня на один узел.
        if node.left:
            queue.append((node.left, depth + 1))
        if node.right:
            queue.append((node.right, depth + 1))

assert min_depth(build_tree([1, 2, 3, 4, None, None, None])) == 2
assert min_depth(build_tree([1, None, 2, None, None, None, 3])) == 3
assert min_depth(build_tree([5])) == 1
assert min_depth(None) == 0
print(min_depth(build_tree([1, 2, 3, 4])))

2


## 4. Произведение минимума и максимума BST
Минимум — крайний левый узел, максимум — крайний правый. Идём по этим веткам. **O(h)** по времени и **O(1)** по памяти.

In [4]:
def min_max_product(arr):
    # Массив должен задавать BST: значения в левом поддереве меньше корня, в правом — больше.
    # Сначала восстанавливаем узлы дерева; build_tree сама свойство BST не проверяет.
    root = build_tree(arr)
    if root is None:
        return None  # У пустого дерева нет минимума и максимума.

    left = right = root  # Независимые указатели для поиска двух крайних узлов.
    # Пока есть левый ребёнок, можно перейти к меньшему значению.
    # В крайнем левом узле окажется минимум всего BST.
    while left.left:
        left = left.left
    # Симметрично: крайний правый узел содержит максимум всего BST.
    while right.right:
        right = right.right

    # Если узел один, минимум и максимум совпадают — получится квадрат его значения.
    # Поиск занимает O(h), но вся функция с построением дерева — O(n) по времени и памяти.
    return left.val * right.val

assert min_max_product([8, 3, 10, 1, 6, None, 14]) == 14
assert min_max_product([5]) == 25
assert min_max_product([]) is None
print(min_max_product([8, 3, 10, 1, 6, None, 14]))

14


## 5. Сравнение двух деревьев
Сравниваем значения узлов и рекурсивно оба поддерева. Деревья равны, если совпадают и значения, и структура. **O(n)** по времени, **O(h)** по памяти.

In [5]:
def is_same_tree(a, b):
    # Один вызов сравнивает два узла и все их потомки в соответствующих позициях.
    if a is None or b is None:
        # Два пустых поддерева равны. Если пусто только одно, структуры уже различаются.
        return a is b
    # Совпадения значений недостаточно: должны совпасть и левые, и правые поддеревья.
    # Здесь сравниваем одноимённые ветви, поскольку требуется равенство, а не зеркальность.
    # and останавливает вычисления при первом False; рекурсия заканчивается на пустых ветвях.
    return (a.val == b.val
            and is_same_tree(a.left, b.left)
            and is_same_tree(a.right, b.right))

assert is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3]))
assert not is_same_tree(build_tree([1, 2]), build_tree([1, None, 2]))
assert not is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 4]))
assert is_same_tree(None, None)
print(is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3])))

True
